# Task 3A: Example Data Analysis across Topic Prevalences
**Group 1**: Percentage Agreement & Bennett, Alpert, and Goldstein’s S

In the real example data, tweets come from three different topics:
- `[BLM]` (Black Lives Matter)
- `[Covid-19]`
- `[Elections2020]`

Each topic naturally has a different prevalence of offensive language. We evaluate how **Percentage Agreement** and **Bennett's S** compare across these different topic prevalences.


In [ ]:
import json
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

# load example files
files = sorted(glob.glob("example_data/*.jsonl"))
if not files:
    files = sorted(glob.glob("/scratch/group_01/bsem-group/annotation_experiment/example_data/*.jsonl"))

data = []
for f in files:
    with open(f, "r", encoding="utf-8") as fp:
        for line in fp:
            line = line.strip()
            if line:
                data.append(json.loads(line))

df = pd.DataFrame(data)

# extract topic from tweet text
def get_topic(text):
    if "[Covid-19]" in text:
        return "Covid-19"
    elif "[Elections2020]" in text:
        return "Elections2020"
    elif "[BLM]" in text:
        return "BLM"
    return "Other"

# extract labels for anna and max
def get_label(anns, user):
    for a in anns:
        if a["user"] == user:
            return "offensive" if "offensive" in a["labels"] else "not offensive"
    return "missing"

df["topic"] = df["text"].apply(get_topic)
df["anna"] = df["labels_annotation"].apply(lambda x: get_label(x, "anna"))
df["max"] = df["labels_annotation"].apply(lambda x: get_label(x, "max"))

print(f"Total samples: {len(df)}")
print("Tweet counts per topic:")
print(df["topic"].value_counts())


## 1. Metrics Formulas
- **Prevalence**: Fraction of tweets that are offensive (where at least one annotator labeled offensive).
- **Percentage Agreement ($P_o$)**: $\frac{\text{agreements}}{N}$
- **Bennett's S**: $S = 2P_o - 1$ (for $k = 2$ categories)


In [ ]:
def evaluate_group(subset, name):
    n = len(subset)
    
    # count offensive tweets (tagged by either anna or max)
    offensive_cases = ((subset["anna"] == "offensive") | (subset["max"] == "offensive")).sum()
    prevalence = offensive_cases / n
    
    # agreement
    agreements = (subset["anna"] == subset["max"]).sum()
    po = agreements / n
    
    # Bennett's S: k = 2
    s = 2.0 * po - 1.0
    
    return {
        "Topic": name,
        "Total (N)": n,
        "Offensive Count": offensive_cases,
        "Prevalence": f"{prevalence:.1%}",
        "Agreements": agreements,
        "Percentage Agreement (Po)": f"{po:.4f} ({po:.1%})",
        "Bennett's S": f"{s:.4f} ({s:.1%})",
        "prevalence_num": prevalence,
        "po_num": po,
        "s_num": s
    }

topics = ["BLM", "Covid-19", "Elections2020"]
results = []

for t in topics:
    sub = df[df["topic"] == t]
    results.append(evaluate_group(sub, t))

# also include all data together
results.append(evaluate_group(df, "All Topics"))

df_results = pd.DataFrame(results)
display_cols = ["Topic", "Total (N)", "Offensive Count", "Prevalence", "Agreements", "Percentage Agreement (Po)", "Bennett's S"]
df_results[display_cols]


## 2. Comparing the Topics by Prevalence and Bennett's S


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

plot_data = df_results[df_results["Topic"] != "All Topics"]
topics_list = plot_data["Topic"]
x = range(len(topics_list))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], plot_data["po_num"], width=width, label="Percentage Agreement ($P_o$)", color="#3498db")
bars2 = ax.bar([i + width/2 for i in x], plot_data["s_num"], width=width, label="Bennett's S ($S$)", color="#e67e22")

ax.set_xticks(list(x))
# include prevalence in x tick labels
ax.set_xticklabels([f"{r['Topic']}\n(Prev: {r['Prevalence']})" for _, r in plot_data.iterrows()], fontsize=11)
ax.set_ylabel("Score (0.0 to 1.0)", fontsize=11)
ax.set_title("Agreement Across Topics with Different Prevalences", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.6)

# add values on top of bars
ax.bar_label(bars1, fmt="%.2f", padding=3, fontsize=10)
ax.bar_label(bars2, fmt="%.2f", padding=3, fontsize=10)

plt.tight_layout()
plt.show()


## 3. Summary & Findings

1. **Exact Metrics by Topic**:
   - **BLM** (Prevalence: **15.6%**): 29 agreements out of 32 tweets -> **$P_o = 90.6\%$**, **Bennett's $S = 0.8125$**.
   - **Covid-19** (Prevalence: **28.6%**): 19 agreements out of 21 tweets -> **$P_o = 90.5\%$**, **Bennett's $S = 0.8095$**.
   - **Elections2020** (Prevalence: **28.6%**): 26 agreements out of 28 tweets -> **$P_o = 92.9\%$**, **Bennett's $S = 0.8571$**.
   - Total across all topics: 74 agreements out of 81 tweets ($P_o = 91.4\%$, $S = 0.8272$).

2. **Key Insight for Group 1 (Why Bennett's S is almost identical across topics)**:
   - Even though offensive prevalence nearly doubled (from 15.6% in BLM to 28.6% in Covid and Elections), **Bennett's $S$ barely changed ($0.81$ vs. $0.81$ vs. $0.86$)**.
   - **Why?** Because Bennett's formula ($S = 2P_o - 1$) depends **only** on the raw percentage agreement ($P_o$). Because Anna and Max maintained a steady ~90% agreement across all three topics, Bennett's $S$ remained virtually unchanged.
   - This demonstrates that Bennett's $S$ is mathematically insensitive to empirical prevalence shifts as long as annotator accuracy stays consistent.
